In [43]:

import os
import tempfile
from cryptography.fernet import Fernet, InvalidToken

# Module-level Fernet instance (as established in validate_credentials)
FERNET_KEY = Fernet.generate_key()
_fernet = Fernet(FERNET_KEY)


In [44]:
KEY_PATH = "./fernet.key"  # store alongside your project; never commit this to git

def load_or_create_fernet() -> Fernet:
    """Load key from disk if it exists, otherwise generate and save it."""
    if os.path.exists(KEY_PATH):
        with open(KEY_PATH, "rb") as f:
            key = f.read()
        print(f"Loaded existing key from {KEY_PATH}")
    else:
        key = Fernet.generate_key()
        with open(KEY_PATH, "wb") as f:
            f.write(key)
        os.chmod(KEY_PATH, 0o600)  # owner read/write only — rw-------
        print(f"Generated new key, saved to {KEY_PATH}")
    return Fernet(key)

_fernet = load_or_create_fernet()


Loaded existing key from ./fernet.key


In [45]:

def generate_filename() -> str:
    """Generate a unique temp file path for credential storage."""
    fd, path = tempfile.mkstemp(suffix=".enc")
    os.close(fd)  # mkstemp opens the file — close the fd, keep the path
    return path
    

In [46]:

def store_credentials(credential: bytes, file) -> bool:
    """
    Write encrypted credential bytes to an already-open file handle.
    The caller is responsible for opening and closing the file.

    Args:
        credential: encrypted bytes from Fernet
        file:       open binary-writable file handle (mode 'w+b')

    Returns:
        True on success, False on write failure
    """
    if not isinstance(credential, bytes):
        raise TypeError(f"Expected bytes, got {type(credential).__name__}")
    try:
        file.write(credential)
        file.flush()
        os.fsync(file.fileno())  # force write to disk, not just OS buffer
        return True
    except OSError as e:
        print(f"Failed to store credentials: {e}")
        return False
        

In [47]:
# --- run ---
credential = {'key': 'test-string'}
key_value = credential.get('key')

if not key_value:
    raise ValueError("Invalid credential: missing 'key' field")

plaintext_bytes = key_value.encode()
encrypted = _fernet.encrypt(plaintext_bytes)
filepath = generate_filename()

try:
    with open(filepath, "wb") as file:         # "wb" — write only, no ambiguous read/write
        success = store_credentials(encrypted, file)
        if not success:
            raise RuntimeError("store_credentials returned False")

    # Verify write — open separately for read
    with open(filepath, "rb") as file:
        stored = file.read()

    assert stored == encrypted, "Stored bytes do not match encrypted credential"
    print(f"SUCCESS — {len(stored)} bytes written to {filepath}")
    print(f"Stored: {stored}")

except (OSError, AssertionError, RuntimeError) as e:
    if os.path.exists(filepath):
        os.remove(filepath)
    raise RuntimeError(f"File storage failed: {e}") from e

SUCCESS — 100 bytes written to /var/folders/5q/q18nbzjj533_v2xn3g6ld2lh0000gp/T/tmplgseuqny.enc
Stored: b'gAAAAABp-QSQWLUdCPghU7GVoPvhZeVOPU9y28iX4_vySxPGfI6azEeDehdED1RoBHwfOvorCSR5Xqp06RGfP1PLht5YV9RkmA=='
